In [ ]:
%%capture
%cd ..

In [ ]:
import os
print("Current working directory:", os.getcwd())

# 3D fMRI Encoding and Decoding

This notebook calls `fmri_taesd.py` and demonstrates a 3D fMRI autoencoder path:

`[B, 16, 80, 96, 80] -> [B, latent_channels, 10, 12, 10] -> [B, 16, 80, 96, 80]`

The model is randomly initialized unless you load your own trained checkpoint.

In [ ]:
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from fmri_taesd import FMRI3DAutoEncoder

dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device", dev)

latent_channels = 8
fmri_ae = FMRI3DAutoEncoder(input_channels=16, latent_channels=latent_channels).to(dev).eval()

# Optional: load a trained checkpoint after you have trained this 3D autoencoder.
# checkpoint = torch.load("fmri_taesd_checkpoint.pth", map_location=dev)
# fmri_ae.load_state_dict(checkpoint)

## Prepare an fMRI tensor

Expected shape is `[B, 16, 80, 96, 80]`. If you already have a `.pt` or `.npy` file, set `fmri_path` below. Otherwise the cell creates a synthetic tensor so the encode/decode path can be tested.

In [ ]:
import numpy as np

fmri_path = None  # Example: "data/sample_fmri.npy" or "data/sample_fmri.pt"

def normalize_fmri(x, eps=1e-6):
    mean = x.mean(dim=(2, 3, 4), keepdim=True)
    std = x.std(dim=(2, 3, 4), keepdim=True).clamp_min(eps)
    return (x - mean) / std

if fmri_path is None:
    fmri = torch.randn(1, 16, 80, 96, 80)
else:
    path = Path(fmri_path)
    if path.suffix == ".pt":
        fmri = torch.load(path, map_location="cpu")
    elif path.suffix == ".npy":
        fmri = torch.from_numpy(np.load(path)).float()
    else:
        raise ValueError("Use a .pt or .npy file, or add a NIfTI loader here.")

if fmri.ndim == 4:
    fmri = fmri.unsqueeze(0)

expected_shape = (1, 16, 80, 96, 80)
if tuple(fmri.shape) != expected_shape:
    raise ValueError(f"Expected {expected_shape}, got {tuple(fmri.shape)}")

fmri = normalize_fmri(fmri.float()).to(dev)
print("fMRI input:", tuple(fmri.shape), fmri.dtype, fmri.device)

## Encode the 3D fMRI volume

In [ ]:
with torch.no_grad():
    fmri_latent = fmri_ae.encode(fmri)

print("Encoded latent:", tuple(fmri_latent.shape))
torch.save(fmri_latent.cpu(), "fmri_encoded_latent.pt")
print("Saved latent to fmri_encoded_latent.pt")

## Decode the latent volume

In [ ]:
with torch.no_grad():
    fmri_recon = fmri_ae.decode(fmri_latent)

print("Decoded reconstruction:", tuple(fmri_recon.shape))
torch.save(fmri_recon.cpu(), "fmri_decoded_reconstruction.pt")
print("Saved reconstruction to fmri_decoded_reconstruction.pt")

## Compare one 2D slice

This visualizes one channel and one depth slice from the original and reconstructed 3D volumes.

In [ ]:
channel = 0
depth = fmri.shape[2] // 2

original_slice = fmri[0, channel, depth].detach().cpu()
recon_slice = fmri_recon[0, channel, depth].detach().cpu()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original_slice, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(recon_slice, cmap="gray")
axes[1].set_title("Decoded")
axes[1].axis("off")
plt.tight_layout()

## Next step for latent diffusion

After the autoencoder is trained, use `fmri_ae.encode(fmri)` to produce compact latent volumes. A 3D diffusion model can then be trained on tensors shaped `[B, latent_channels, 10, 12, 10]`, and generated latents can be passed through `fmri_ae.decode(latents)` for 3D fMRI reconstruction.